# ProvideQ Web RAG Evaluation

This notebook reads `outputs/*/results.csv` and compares configurations that were evaluated on exactly the same question IDs.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

OUTPUT_DIR = Path('../outputs') if Path('../outputs').exists() else Path('outputs')


In [ ]:
runs = []
for csv_path in sorted(OUTPUT_DIR.glob('*/results.csv')):
    parts = csv_path.parent.name.rsplit('_', 2)
    if len(parts) != 3:
        continue
    retrieval, reranker, evaluation = parts
    frame = pd.read_csv(csv_path)
    frame['score'] = pd.to_numeric(frame['score'], errors='coerce')
    runs.append({
        'retrieval': retrieval,
        'reranker': reranker,
        'evaluation': evaluation,
        'label': f'{retrieval} + {reranker}',
        'path': csv_path,
        'frame': frame,
        'question_ids': tuple(frame['question_id'].astype(str)),
    })

summary = pd.DataFrame([
    {
        'retrieval': run['retrieval'],
        'reranker': run['reranker'],
        'evaluation': run['evaluation'],
        'questions': len(run['frame']),
        'scored_questions': run['frame']['score'].notna().sum(),
        'mean_score': run['frame']['score'].mean(),
    }
    for run in runs
])
summary.sort_values(['evaluation', 'mean_score'], ascending=[True, False])


In [ ]:
def plot_evaluation(layer):
    layer_runs = [run for run in runs if run['evaluation'] == layer]
    if not layer_runs:
        print(f'No {layer} results found.')
        return

    reference_ids = layer_runs[0]['question_ids']
    mismatched = [run['label'] for run in layer_runs if run['question_ids'] != reference_ids]
    if mismatched:
        raise ValueError(
            f'{layer} runs do not use the same questions. Re-run them with the same '
            f'--num-questions and --seed. Mismatched: {mismatched}'
        )

    values = pd.Series(
        {run['label']: run['frame']['score'].mean() for run in layer_runs}
    ).sort_values(ascending=False)

    plt.figure(figsize=(9, 5))
    values.plot(kind='bar')
    plt.title(f'{layer.capitalize()} evaluation')
    plt.xlabel('Paperclip ranking + reranker')
    plt.ylabel('Mean score')
    plt.xticks(rotation=30, ha='right')
    plt.ylim((-1, 1) if layer == 'judge' else (0, 1))
    plt.tight_layout()
    plt.show()


## Lexical evaluation


In [ ]:
plot_evaluation('lexical')


## Semantic evaluation


In [ ]:
plot_evaluation('semantic')


## LLM-as-a-judge evaluation


In [ ]:
plot_evaluation('judge')
